# hoodini CLI Workflow Demo

This notebook runs all main parts of the `hoodini` CLI workflow, step by step, as implemented in the command line tool. Each cell corresponds to a logical section of the pipeline.

In [1]:
# Patch asyncio for Jupyter compatibility
import nest_asyncio
nest_asyncio.apply()
print("Asyncio patched for Jupyter compatibility.")

Asyncio patched for Jupyter compatibility.


## 1. Import Required Libraries
Import all necessary libraries and modules used in cli.py, including any custom modules.

In [2]:
import os
import sys
import logging
import pandas as pd
import rich_click as click
import tomli
from pathlib import Path
from hoodini.config import load_default_config
from hoodini.utils.cli_helpers import MutuallyExclusiveOption
from hoodini.utils.validation import validate_input_file, validate_domains
from hoodini.utils.logging_utils import console, logger, stage_header, stage_done, run_with_spinner

## 2. Parse Command-Line Arguments
Simulate or manually define the command-line arguments that would be passed to cli.py, and parse them as needed.

In [3]:
# Initialize console and logger (these are already available as imports)
# Test logging
logger.info("Hoodini workflow notebook initialized")

## 3. Initialize Main Workflow Components
Set up any objects, configurations, or initial states required for the workflow.(THIS IS OPTIONAL)

In [4]:
class ConfigObj(dict):
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__

# Initialize configuration and logging
config = load_default_config()

# Display loaded config for reference
console.print('Loaded configuration:', config)

config = ConfigObj(config)

Loaded configuration:
{
    'output': 'results',
    'num_threads': 10,
    'max_concurrent_downloads': 8,
    'apikey': '',
    'mod': 'win_nts',
    'wn': 20000,
    'height_factor': 20,
    'ngenes': 10,
    'minwin': 2000,
    'minwin_type': 'both',
    'tree_mode': 'fast_ml',
    'tree_file': 'target_prots.nwk',
    'nj_algorithm': 'nj',
    'aai_mode': 'wgrr',
    'aai-subset-mode': 'target_region',
    'ani_mode': 'fastani',
    'cand_mode': 'best_id',
    'clust_method': 'diamond_deepclust',
    'padloc': False,
    'deffinder': False,
    'ncrna': False,
    'cctyper': False,
    'domains': '',
    'domains_metadata': '',
    'min_prevalence': 0.0,
    'genomad': False,
    'antidefense': False,
    'phrogs': False,
    'sorfs': False,
    'prot_links': False,
    'nt_links': False,
    'nt_aln_mode': 'blastn',
    'assembly_folder': '',
    'assembly_db': '/env/db/hoodini',
    'img_db': '',
    'img_nuc': '',
    'blast': '',
    'img': '/mnt/fastdb/imgpr_imgvr/data_structure/',
    'img_metadata': '/mnt/fastdb/imgpr_imgvr/merged_metadata/imgvr_pr_taxids.txt',
    'keep': False,
    'force': False
}

## 4. Execute Core Workflow Logic
Run the main logic of the workflow, step by step, as implemented in cli.py.

## Initialize inputs

In [5]:

# Run main workflow logic (simplified for notebook)
# 1. Parse input files
console.print(f'Parsing input file: {config.input_file}')


from hoodini.pipeline.initialize import initialize_inputs

# Initialize the records DataFrame
records = initialize_inputs(
    input_path="cas9.txt",
    output=config.output,
)

Parsing input file: None

📁 Assembly DB found: /home/pentamorfico/software/hoodini/src/hoodini/data/assembly_summary.parquet (updated: 
2025-12-23)

✅ Using existing database (less than 1 month old).

⚠️  Folder results already exists.

⌨️  Remove it? (y/N) (no):

✔️  Created folder results.

In [6]:
from hoodini.pipeline.parse_ipg import run_ipg

records = run_ipg(
    records_df=records,  # or provide a DataFrame if you already have one
    cand_mode="best_id",)

console.print("IPG step complete. Records loaded.")

🔍  Fetching IPG data...

Output()

[DEBUG] Fetching IPG for 10 accessions. First 5: ['WP_198985183.1', 'WP_262856654.1', 'WP_239738697.1', 'WP_217844005.1', 'WP_107544007.1']


[15:21:47] Fetched 20 IPG records for 10 proteins.                                                 ]8;id=331524;file:///home/pentamorfico/software/hoodini/src/hoodini/pipeline/parse_ipg.py\parse_ipg.py]8;;\:]8;id=28456;file:///home/pentamorfico/software/hoodini/src/hoodini/pipeline/parse_ipg.py#143\143]8;;\

🔍  Fetching nucleotide data...

✅  Selecting best IPG records...

IPG step complete. Records loaded.

In [7]:
from hoodini.pipeline.parse_assemblies import run_assembly_parser
result = run_assembly_parser(
    records_df=records,
    output_dir=config.output,
    assembly_folder=config.assembly_folder,
    ncrna=config.ncrna,
    cctyper=config.cctyper,
    genomad=config.genomad,
    blast=config.blast,
    apikey=config.apikey,
    max_concurrent_downloads=config.max_concurrent_downloads,
    img=config.img,
    num_threads=config.num_threads,
    mod=config.mod,
    wn=config.wn,
    sorfs=config.sorfs,
    minwin=config.minwin,
    minwin_type=config.minwin_type,
)

✔️  Downloaded or located assemblies (folder: results/assembly_folder)

{'unique_id': '0', 'assembly_id': 'GCF_035781455.1', 'assembly_level': 'Contig', 'end': 281312, 'faa_path': None, 'failed': None, 'fna_path': None, 'gbf_path': '/home/pentamorfico/software/hoodini/example/results/assembly_folder/GCF_035781455.1/genomic.gbff', 'gff_path': None, 'group': 'bacteria', 'img': None, 'infraspecific_name': 'strain=228', 'input_type': 'protein', 'ipg_id': 468491264, 'ipg_protein_id': 'WP_217844005.1', 'is_refseq_query': True, 'nucleotide_id': 'NZ_JALKNF010000002.1', 'nucleotide_id_no_prefix': 'JALKNF010000002.1', 'og_index': 0, 'organism': 'Mammaliicoccus sciuri', 'organism_name': 'Mammaliicoccus sciuri', 'premade': False, 'protein name': 'type II CRISPR RNA-guided endonuclease Cas9', 'protein_id': 'WP_217844005.1', 'query_protein_id': 'WP_217844005.1', 'sequence_length': '449893', 'source': 'RefSeq', 'species_taxid': 1296, 'start': 278151, 'strain': '228', 'strand': '-', 'taxid': 1296, 'uniprot_id': None, 'nuc_gca_no_prefix': None, 'nuc_gca_full': 'JALKNF01000

Parsing GBFF:   0%|          | 0/10 [00:00<?, ?it/s]

3 contigs below min window size: ['2', '7', '1']

✔️  Extracted neighborhoods and wrote output files.

In [8]:
records = result["records"]
all_gff = result["all_gff"]
all_prots = result["all_prots"]
all_neigh = result["all_neigh"]
valid_uids = result["valid_uids"]

In [9]:
all_prots

protein_id,id,sequence,product,unique_id,target_prot,target_nuc
str,str,str,str,str,str,str
"""WP_347132630.1""","""WP_347132630.1""","""MKRNYILGLDIGITSVGYGIIDYETRDVID…","""type II CRISPR RNA-guided endo…","""1""","""WP_347132630.1""","""NZ_JBDLKV010000051.1"""
"""WP_002479864.1""","""WP_002479864.1""","""MEENNSFDFSNVWRIIKRYLKWLIIFPIAG…","""YveK family protein""","""5""","""WP_107596301.1""","""NZ_PZGB01000021.1"""
"""WP_002479865.1""","""WP_002479865.1""","""MVDAANAKKGLTYKIINMNFDNTMLAHRLS…","""FeoA family protein""","""5""","""WP_107596301.1""","""NZ_PZGB01000021.1"""
"""WP_023015504.1""","""WP_023015504.1""","""MSNTYCIVGNPNVGKTTLFNALTGSYEYVG…","""ferrous iron transport protein…","""5""","""WP_107596301.1""","""NZ_PZGB01000021.1"""
"""WP_002479867.1""","""WP_002479867.1""","""MTLLINLVIFLLIFGYAAFTIVRFFKKSKQ…","""FeoB-associated Cys-rich membr…","""5""","""WP_107596301.1""","""NZ_PZGB01000021.1"""
…,…,…,…,…,…,…
"""WP_262588089.1""","""WP_262588089.1""","""MLSKEAESFLMKLRITLMERGKDDQSINEI…","""hypothetical protein""","""6""","""WP_262588072.1""","""NZ_CP095096.1"""
"""WP_114604511.1""","""WP_114604511.1""","""MTISNQMMKGLLDGAILALIAQGETYGYEI…","""PadR family transcriptional re…","""6""","""WP_262588072.1""","""NZ_CP095096.1"""
"""WP_105994655.1""","""WP_105994655.1""","""MTLLINGIILLLILGYAAFTIIRFFKKSKQ…","""FeoB-associated Cys-rich membr…","""6""","""WP_262588072.1""","""NZ_CP095096.1"""


In [10]:
from hoodini.pipeline.cluster_proteins import cluster_proteins  # assuming it's saved here

cluster = cluster_proteins(
    all_prots,
    output_dir=config.output,
    clust_method=config.clust_method,
    sorfs=config.sorfs
)

✔️       Protein clustering complete

In [11]:
cluster

protein_id,id,sequence,product,unique_id,target_prot,target_nuc,fam_cluster
str,str,str,str,str,str,str,i64
"""WP_347132630.1""","""WP_347132630.1""","""MKRNYILGLDIGITSVGYGIIDYETRDVID…","""type II CRISPR RNA-guided endo…","""1""","""WP_347132630.1""","""NZ_JBDLKV010000051.1""",1
"""WP_002479864.1""","""WP_002479864.1""","""MEENNSFDFSNVWRIIKRYLKWLIIFPIAG…","""YveK family protein""","""5""","""WP_107596301.1""","""NZ_PZGB01000021.1""",null
"""WP_002479865.1""","""WP_002479865.1""","""MVDAANAKKGLTYKIINMNFDNTMLAHRLS…","""FeoA family protein""","""5""","""WP_107596301.1""","""NZ_PZGB01000021.1""",30
"""WP_023015504.1""","""WP_023015504.1""","""MSNTYCIVGNPNVGKTTLFNALTGSYEYVG…","""ferrous iron transport protein…","""5""","""WP_107596301.1""","""NZ_PZGB01000021.1""",34
"""WP_002479867.1""","""WP_002479867.1""","""MTLLINLVIFLLIFGYAAFTIVRFFKKSKQ…","""FeoB-associated Cys-rich membr…","""5""","""WP_107596301.1""","""NZ_PZGB01000021.1""",35
…,…,…,…,…,…,…,…
"""WP_262588089.1""","""WP_262588089.1""","""MLSKEAESFLMKLRITLMERGKDDQSINEI…","""hypothetical protein""","""6""","""WP_262588072.1""","""NZ_CP095096.1""",24
"""WP_114604511.1""","""WP_114604511.1""","""MTISNQMMKGLLDGAILALIAQGETYGYEI…","""PadR family transcriptional re…","""6""","""WP_262588072.1""","""NZ_CP095096.1""",11
"""WP_105994655.1""","""WP_105994655.1""","""MTLLINGIILLLILGYAAFTIIRFFKKSKQ…","""FeoB-associated Cys-rich membr…","""6""","""WP_262588072.1""","""NZ_CP095096.1""",35


In [12]:
from hoodini.pipeline.taxonomy import parse_taxonomy_and_build_tree

tree_str, den_data = parse_taxonomy_and_build_tree(
    records=records,
    all_gff=all_gff,
    all_neigh=all_neigh,
    all_prots=all_prots,
    output_dir=config.output,
    tree_mode=config.tree_mode,
    tree_file=config.tree_file,
    num_threads=config.num_threads,
    aai_mode=config.aai_mode,
    ani_mode=config.ani_mode,
    aai_subset_mode=config.aai_subset_mode,
    nj_algorithm=config.nj_algorithm,
)

/home/pentamorfico/miniforge3/envs/hoodini/lib/python3.12/site-packages/UniProtMapper/utils.py:15: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
FAMSA (Fast and Accurate Multiple Sequence Alignment) 
  version 2.4.1-45c9b2b (2025-05-09)
  S. Deorowicz, A. Debudaj-Grabysz, A. Gudys

Done!


✔️       Tree saved as Newick

In [13]:
from hoodini.pipeline.protein_links import run_protein_links
pairwise_aa = run_protein_links(
    output_dir=config.output,
    all_prots=all_prots,
    threads=config.num_threads,
    evalue=1e-5
)

Building Diamond database...

Diamond database built: results/results.dmnd

Running all-vs-all blastp (Diamond)...

No protein pairwise comparisons found

In [14]:
from hoodini.pipeline.pairwise_nt import run_pairwise_nt

pairwise_ani, nt_links = run_pairwise_nt(
    all_neigh=all_neigh,
    all_gff=all_gff,
    output_dir=config.output,
    nt_aln_mode="blastn",
    ani_mode="blastn",
    nt_links=True,
    threads=config.num_threads,
)

[15:22:09] Making BLAST DB at results/db                                                         ]8;id=545308;file:///home/pentamorfico/software/hoodini/src/hoodini/pipeline/pairwise_nt.py\pairwise_nt.py]8;;\:]8;id=366656;file:///home/pentamorfico/software/hoodini/src/hoodini/pipeline/pairwise_nt.py#559\559]8;;\

           Running BLAST: blastn -query results/neighborhood/neighborhoods.fasta -db results/db  ]8;id=816199;file:///home/pentamorfico/software/hoodini/src/hoodini/pipeline/pairwise_nt.py\pairwise_nt.py]8;;\:]8;id=49756;file:///home/pentamorfico/software/hoodini/src/hoodini/pipeline/pairwise_nt.py#572\572]8;;\
           -task blastn -evalue 1e-05 -soft_masking false -dust no -outfmt 6 qseqid sseqid                         
           pident length mismatch gapopen qstart qend sstart send evalue bitscore qlen slen                        
           -num_threads 10                                                                                         

[15:22:10] Raw BLAST HSPs (unique unordered, no self): 435                                       ]8;id=516058;file:///home/pentamorfico/software/hoodini/src/hoodini/pipeline/pairwise_nt.py\pairwise_nt.py]8;;\:]8;id=356010;file:///home/pentamorfico/software/hoodini/src/hoodini/pipeline/pairwise_nt.py#591\591]8;;\

In [15]:
pairwise_ani

Ref_name,Query_name,ANI,Align_fraction_ref,Align_fraction_query
str,str,f64,f64,f64
"""NZ_JAELVP010000003.1""","""MPNR01000008.1""",85.164,25.145,25.699
"""NZ_PZGB01000021.1""","""NZ_JAIEXF010000005.1""",77.581,56.444,60.279
"""NZ_JAIEXF010000005.1""","""MPNR01000008.1""",74.331,15.421,15.756
"""NZ_PZGB01000021.1""","""MPNR01000008.1""",80.393,14.005,15.162
"""NZ_JAIEXF010000005.1""","""NZ_JAELVP010000003.1""",73.463,15.699,15.722
…,…,…,…,…
"""NZ_CP095096.1""","""MPNR01000008.1""",77.33,10.057,11.011
"""NZ_PZGB01000021.1""","""NZ_CP095096.1""",81.287,88.525,90.964
"""NZ_CP095096.1""","""NZ_JAELVP010000003.1""",75.905,9.995,11.941


In [16]:
nt_links

query,query_start,query_end,ref,ref_start,ref_end,ani
str,i64,i64,str,i64,i64,f64
"""NZ_JAOPKZ010000017.1""",35247,38697,"""MPNR01000008.1""",26655,30097,88.094
"""NZ_JAOPKZ010000017.1""",34188,34476,"""MPNR01000008.1""",23525,23814,74.15
"""NZ_JAOPKZ010000017.1""",34188,34476,"""MPNR01000008.1""",24055,24344,73.737
"""NZ_JAOPKZ010000017.1""",34188,34476,"""MPNR01000008.1""",23657,23947,73.244
"""NZ_JAOPKZ010000017.1""",34188,34476,"""MPNR01000008.1""",24187,24475,73.443
…,…,…,…,…,…,…
"""NZ_PZGB01000021.1""",28566,28784,"""NZ_JALKNF010000002.1""",275669,275888,71.491
"""NZ_PZGB01000021.1""",28623,28915,"""NZ_JALKNF010000002.1""",275594,275887,68.608
"""NZ_PZGB01000021.1""",28820,29112,"""NZ_JALKNF010000002.1""",275399,275689,69.381


In [17]:
from hoodini.extra_tools.domain import run_domain
# config.domains is already a validated list of database names
config.domains = ["pfam"]
domains_data = run_domain(all_prots, config.output, config.domains, config.num_threads)

🔍      Annotating domains for pfam with HMMER...

✔️       Domain annotation complete: 335 matches from 1 databases (deduplicated from 652).

In [18]:
from hoodini.extra_tools.padloc import run_padloc
padloc_df = run_padloc(all_gff, all_prots, config.output, config.num_threads)
all_prots = all_prots.join(padloc_df, on="id", how="left")


🧬      Running PADLOC...

[15:22:36] >> Scanning temp.fasta for defence system proteins
[15:23:01] >> Searching temp.fasta for defence systems


Warning messages:
1: package ‘tidyverse’ was built under R version 4.3.3 
2: package ‘ggplot2’ was built under R version 4.3.3 
3: package ‘tibble’ was built under R version 4.3.3 
4: package ‘tidyr’ was built under R version 4.3.3 
5: package ‘readr’ was built under R version 4.3.3 
6: package ‘purrr’ was built under R version 4.3.3 
7: package ‘dplyr’ was built under R version 4.3.3 
8: package ‘stringr’ was built under R version 4.3.3 
9: package ‘forcats’ was built under R version 4.3.3 
10: package ‘lubridate’ was built under R version 4.3.3 


[15:23:54] >> Writing output to '/home/pentamorfico/software/hoodini/example/results/padloc/temp.fasta_padloc.csv'


Error: Cannot open file for writing:
* '/home/pentamorfico/software/hoodini/example/results/padloc/temp.fasta_padloc.csv'
Execution halted
[15:23:54] ERROR >> errexit on line 425


CalledProcessError: Command '['padloc', '--faa', 'results/temp.fasta', '--gff', 'results/temp.gff', '--cpu', '10', '--outdir', 'results/padloc']' returned non-zero exit status 1.

In [ ]:
from hoodini.extra_tools.defensefinder import run_defensefinder
deffinder_df = run_defensefinder(all_gff, all_prots, config.output)
all_prots = all_prots.join(deffinder_df, on="id", how="left")

In [ ]:

from hoodini.extra_tools.blast import run_blast
blast_data = run_blast(all_neigh, config.output, "query.fa", config.num_threads, valid_uids)
#convert to GFF-like format in which start = qstart, end=qend, ID=qseqid, feature type should be "region"
if not blast_data.empty:
    gff_df = pd.DataFrame({
        "seqid": blast_data["seqid"],
        "source": "hoodini",
        "type": "region",
        "start": blast_data["start"],
        "end": blast_data["end"],
        "score": ".",
        "strand": "+",
        "phase": ".",
        "attributes": "ID=" + blast_data["nc_feature"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)


from hoodini.extra_tools.cctyper import run_cctyper
cctyper_df, crispr_df = run_cctyper(all_gff, all_prots, all_neigh, config.output, config.num_threads, valid_uids)
#if cctyper_df is not empty:
if not cctyper_df.empty:
    all_prots = all_prots.merge(cctyper_df, on="id", how="left")
if not crispr_df.empty:
    gff_df = pd.DataFrame({
        "seqid": crispr_df["Contig"],
        "source": "hoodini",
        "type": "region",
        "start": crispr_df["start"],
        "end": crispr_df["end"],
        "score": ".",
        "strand": ".",
        "phase": ".",
        "attributes": "ID=" + crispr_df["nc_feature"] + ";"
    })
    #append to all_gff
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)

from hoodini.extra_tools.genomad import run_genomad
genomad_df = run_genomad(all_neigh, config.output, config.num_threads, valid_uids)
if not genomad_df.empty:
    gff_df = pd.DataFrame({
        "seqid": genomad_df["seqid"],
        "source": "hoodini",
        "type": "region",
        "start": genomad_df[["start", "end"]].min(axis=1),
        "end": genomad_df[["start", "end"]].max(axis=1),
        "score": ".",
        "strand": ".Z",
        "phase": ".",
        "attributes": "ID=" + genomad_df["mge_type"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)


from hoodini.extra_tools.ncrna import run_ncrna
ncrna_data = run_ncrna(all_neigh, den_data, config.output, config.num_threads, valid_uids)
if not ncrna_data.empty:
    gff_df = pd.DataFrame({
        "seqid": ncrna_data["nucid"],
        "source": "hoodini",
        "type": "ncRNA",
        "start": ncrna_data[["start", "end"]].min(axis=1),
        "end": ncrna_data[["start", "end"]].max(axis=1),
        "score": ".",
        "strand": ncrna_data["strand_ncrna"],
        "phase": ".",
        "attributes": "ID=" + ncrna_data["nc_feature"] + ";"
    })
    all_gff = pd.concat([all_gff, gff_df], ignore_index=True)

In [ ]:
# Use the configured output directory directly
outdir = Path(config.output+ "/hoodini-viz")
outdir.mkdir(parents=True, exist_ok=True)

# GFF
all_gff.to_csv(outdir / "defaultGFF.gff", index=False, sep="\t", header=False)

# Baselines
all_neigh = all_neigh.rename(columns={
    "unique_id": "hood_id",
    "start_win": "start",
    "end_win": "end",
    "target_prot": "align_gene",
})
all_neigh[["hood_id", "seqid", "start", "end", "align_gene"]].to_csv(
    outdir / "defaultBaselines.txt", sep="\t", index=False
)

# Protein metadata
all_prots = all_prots.rename(columns={"protein_id": "gene_id", "fam_cluster": "cluster"})
all_prots["product"] = all_prots["product"].str.replace("\n", " ")
all_prots.to_csv(outdir / "defaultProteinMetadata.txt", sep="\t", index=False)

# Tree metadata
den_data = den_data.rename(columns={"unique_id": "leaf_id"})
den_data["leaf_id"] = den_data["leaf_id"].astype(str).str.extract(r"(\d+)").astype(int)
den_data.to_csv(outdir / "defaultTreeMetadata.txt", sep="\t", index=False)

# Newick tree
with open(outdir / "defaultNewick.txt", "w", encoding="utf-8") as f:
    f.write(tree_str)
    
# Domain metadata
if config.domains:
    with open(outdir / "defaultDomains.txt", "w", encoding="utf-8") as f:
        domains_data[["protein_id","domain_id","start","end","database"]].to_csv(f, sep="\t", index=False, header=False)
    
# Nucleotide link
# write nucleotide links; if nt_links not produced, write placeholder header-only file
if nt_links:
    nt_links[["query","query_start","query_end","ref","ref_start","ref_end","ani"]].to_csv(outdir / "defaultNucleotideLinks.txt", sep="\t", index=False,header=False)
else:
    pd.DataFrame(columns=["query","query_start","query_end","ref","ref_start","ref_end","ani"]).to_csv(outdir / "defaultNucleotideLinks.txt", sep="\t", index=False,header=False)

# Protein links
# pairwise_aa is only created when protein comparisons were run; if missing,
# write an empty placeholder file with the expected columns so downstream
# consumers always find the file.
if pairwise_aa:
    pairwise_aa[["qseqid","sseqid","pident"]].to_csv(outdir / "defaultProteinLinks.txt", sep="\t", index=False,header=False)
else:
    pd.DataFrame(columns=["qseqid","sseqid","pident"]).to_csv(outdir / "defaultProteinLinks.txt", sep="\t", index=False,header=False)

